# Frozen Lake: Monte Carlo Methods

## Introduction

This notebook shows how to use simple MC methods to Gymnasium's Frozen lake environment, a small toy example for stochastic actions. 
Frozen lake involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. The player may not always move in the intended direction due to the slippery nature of the frozen lake.

Here is the [environment description.](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)

The size of the problem, the randomness and the reward structure can be adjusted.

## Setup

You need:
* Gymnasium (see [Installation Instructions](../common/Setup_Gymnasium.ipynb))
* Patched `gym-classics-1.0.0+internal.rev1` or later (see [Installation instructions](../common/Setup_patched_gym_classics.ipynb))

In [1]:
import numpy as np
np.set_printoptions(precision=2)

In [2]:
import gymnasium as gym
import gym_classics
gym_classics.register('gymnasium')

In [3]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")

## A Simple Example

### Create the Environment

**Note:** `success_rate` makes the ground slippery leading to stochastic behavior! 

In [4]:
from gymnasium_display_recorder import VideoWrapper, show

env = gym.make('FrozenLake-v1', 
               map_name="4x4", # can also be "8x8"
               is_slippery=True,
               success_rate=9.0/10.0,
               reward_schedule=(1, 0, 0),
               render_mode="rgb_array")

env_record = VideoWrapper(env, 'FL', render_fps=2)

Videos already exist, I remove them first!


/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/MC/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


We need simple code to run an episode.

In [5]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    Return = 0
    # run one episode
    for i in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        # step: execute an action in the environment
        observation_p, reward, terminated, truncated, info = env.step(action)

        if verbose:
            print (f"Step {i+1}: Obs {observation} -> Action {action} - > Reward {reward}, Obs' {observation_p}")

        observation = observation_p
        Return += reward

        # render the environment
        if render:
            env.render()

        if terminated:
            break
  
    if verbose:
        print(f"Episode Return: {Return}")
    
    return reward

### Random Agent

Check with am agent that moves randomly.

In [6]:
def random_agent_function(observation): 
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return env.action_space.sample()

In [7]:
run_episode(random_agent_function, env_record, max_steps = 100, verbose = True)
show(env_record)

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Step 1: Obs 0 -> Action 2 - > Reward 0, Obs' 1
Step 2: Obs 1 -> Action 2 - > Reward 0, Obs' 2
Step 3: Obs 2 -> Action 0 - > Reward 0, Obs' 1
Step 4: Obs 1 -> Action 3 - > Reward 0, Obs' 1
Step 5: Obs 1 -> Action 1 - > Reward 0, Obs' 5
Episode Return: 0
Showing: ./videos/video_FL-episode-0.mp4


## Use MC Control with Exploring Starts 

We use here the implementation in `gym-classics`.

### Learn A Policy

In [16]:
from gym_classics.algorithms.MC import MC_control_ES, MC_prediction

pol, Q = MC_control_ES(env, discount=1, n = 20000, max_episode_len= 30, verbose = False)
pol

array([2, 3, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 2, 2, 2, 2])

Let's make the policy more readable and follow the layout of the problem.

In [17]:
def decode_action(action):
    return ['←','↓','→','↑'][int(action)]

def decode_policy(policy):
    return np.array([decode_action(a) for a in policy])

In [18]:
decode_policy(pol).reshape(4,4)

array([['→', '↑', '↓', '←'],
       ['←', '→', '↓', '→'],
       ['→', '→', '↓', '←'],
       ['→', '→', '→', '→']], dtype='<U1')

Note that the agent sometimes runs into the wall to avoid the chance of falling into the lake.

MC Control returns the estimated Q-function. We can extract the value function to see the value for each state.

In [20]:
V = np.max(Q, axis = 1)
V.reshape(4,4)

array([[0.51, 0.54, 0.75, 0.64],
       [0.55, 0.57, 0.79, 0.56],
       [0.53, 0.8 , 0.9 , 0.56],
       [0.54, 0.52, 0.98, 0.56]])

### Predict the Value Function

We can also use MC prediction (implemented in `gym-classics`). 

In [21]:
V = MC_prediction(env, pol, discount = 1, n =1000, verbose = False)
V.reshape(4,4)

array([[0.85, 0.85, 0.86, 0.78],
       [0.8 ,  nan, 0.87,  nan],
       [0.81, 0.91, 0.94,  nan],
       [ nan, 1.  , 1.  ,  nan]])

Note that the samples always start from the start state and only follow the policy, so many states have no estimate. 

### Experiment with the Learned Policy

In [22]:
def policy_agent_function_generator(policy):
    def agent_function(obs):
        return policy[obs]
    return agent_function

In [23]:
policy_agent_function = policy_agent_function_generator(pol)

for i in range(10):
    G = run_episode(policy_agent_function, env_record, max_steps = 100, verbose = False)
    print(f"Episode {i}: Retrun={G}")
    show(env_record)


Episode 0: Retrun=1
Showing: ./videos/video_FL-episode-11.mp4


Episode 1: Retrun=1
Showing: ./videos/video_FL-episode-12.mp4


Episode 2: Retrun=1
Showing: ./videos/video_FL-episode-13.mp4


Episode 3: Retrun=1
Showing: ./videos/video_FL-episode-14.mp4


Episode 4: Retrun=1
Showing: ./videos/video_FL-episode-15.mp4


Episode 5: Retrun=0
Showing: ./videos/video_FL-episode-16.mp4


Episode 6: Retrun=1
Showing: ./videos/video_FL-episode-17.mp4


Episode 7: Retrun=1
Showing: ./videos/video_FL-episode-18.mp4


Episode 8: Retrun=1
Showing: ./videos/video_FL-episode-19.mp4


Episode 9: Retrun=1
Showing: ./videos/video_FL-episode-20.mp4


Perform evaluation with 100 random runs.

In [24]:
def policy_agent_function(obs):
    return pol[obs]

Gs = []

for i in range(100):
    G = run_episode(policy_agent_function, env, max_steps = 100, verbose = False)
    Gs.append(G)
   
print(Gs) 

print (f"Success rate: {np.mean(Gs)*100}%")

[1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Success rate: 80.0%


The success rate is rather high since we use a very small problem and a very high success rate.


&copy; 2025 [Michael Hahsler](http://michael.hahsler.net). 
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)